<a href="https://colab.research.google.com/github/AbdullahRasheed452/ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os, subprocess

REPO_URL = "https://github.com/AbdullahRasheed452/ML-Internship"
REPO_DIR = "ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if os.getcwd().split("/")[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

In [17]:
%pip -q install -U duckdb huggingface_hub

In [18]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF_TOKEN')

In [19]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [20]:
# Method: Random Forest
# It builds on the decision tree from Notebook 02, but combines many trees
# so it can catch more patterns than one simple hand written rule
# It also shows which features matter most
# In the starter data, random forest beat the hand rule by a wide margin
# 0.74 vs 0.24 on Precision@50

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [21]:
# Split: grouped by client
# Pages from the same client can look similar, so if the same client
# shows up in both train and test, the model looks better than it really is
# A grouped split keeps each client fully in either train or test, never both
# This gives a more honest test of how the model handles a brand new client

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
# Build features using only the first half of March, so nothing about
# the second half (which defines our label) leaks into the features
q = f"""
SELECT
    f.content_hash_id,
    f.client_hash_id,
    AVG(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_impressions END) AS avg_impressions_first_half,
    AVG(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
    AVG(CASE WHEN f.report_date < '2026-03-16' AND f.gsc_impressions > 0
             THEN f.gsc_clicks * 1.0 / f.gsc_impressions END) AS avg_ctr_first_half,
    ANY_VALUE(c.word_count) AS word_count,
    ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-15')) AS content_age_days,
    SUM(CASE WHEN f.report_date < '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
    SUM(CASE WHEN f.report_date >= '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_second_half
FROM {TABLES['fact_daily']} f
JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
GROUP BY f.content_hash_id, f.client_hash_id
HAVING imp_first_half >= 10
"""

model_data = con.sql(q).df()

model_data['pct_change'] = (model_data['imp_second_half'] - model_data['imp_first_half']) / model_data['imp_first_half']
model_data['is_declining'] = (model_data['pct_change'] < -0.2).astype(int)

model_data = model_data.dropna(subset=['avg_impressions_first_half', 'avg_position_first_half', 'avg_ctr_first_half', 'word_count', 'content_age_days'])

print(len(model_data), 'pages,', model_data['client_hash_id'].nunique(), 'clients')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79616 pages, 41 clients


In [23]:
# Split the data by client, so the same client never appears in both
# train and test, this gives an honest test on clients the model has
# not seen before
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

features = ['avg_impressions_first_half', 'avg_position_first_half', 'avg_ctr_first_half', 'word_count', 'content_age_days']
X = model_data[features]
y = model_data['is_declining']
groups = model_data['client_hash_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print('train clients:', model_data.iloc[train_idx]['client_hash_id'].nunique())
print('test clients:', model_data.iloc[test_idx]['client_hash_id'].nunique())

# Train a random forest on the training clients only
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

train clients: 28
test clients: 13


RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [24]:
# Score the test set with the model
model_scores = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

import numpy as np
import pandas as pd

# Baseline score, same idea as Week 4, gap between page ctr and tier average ctr
test_data = model_data.iloc[test_idx].copy()
test_data['position_bucket'] = pd.cut(
    test_data['avg_position_first_half'],
    bins=[-1, 3, 10, 20, 1000],
    labels=['1_top3', '2_top10', '3_top20', '4_below20']
)
tier_avg = test_data.groupby('position_bucket', observed=True)['avg_ctr_first_half'].transform('mean')
test_data['baseline_score'] = tier_avg - test_data['avg_ctr_first_half']

for k in (20, 50):
    base_p = precision_at_k(test_data['baseline_score'], y_test.values, k)
    model_p = precision_at_k(model_scores, y_test.values, k)
    print(f"Precision@{k}: baseline {base_p:.3f}  vs  model {model_p:.3f}")

Precision@20: baseline 0.300  vs  model 0.650
Precision@50: baseline 0.260  vs  model 0.700


In [25]:
# Model vs baseline, same test clients, same metric

# Precision@20: baseline 0.300  vs  model 0.650
# Precision@50: baseline 0.260  vs  model 0.700

# The model clearly beats the baseline at both cutoffs
# Earlier I had a leakage bug where avg_impressions used the whole month,
# including the second half that the label is built from, which gave a
# fake precision near 1.0
# After fixing this to use only first half of March data as features,
# the results dropped to a believable range, and the model still beats
# the baseline by a solid margin

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
# What does the model rely on most
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
importances

,0
avg_position_first_half,0.236179
word_count,0.229833
avg_impressions_first_half,0.215785
content_age_days,0.202983
avg_ctr_first_half,0.115219


In [27]:
# Look at a few pages the model got wrong, both false positives and false negatives
test_data['model_score'] = model_scores
test_data['actual'] = y_test.values
test_data['predicted'] = (test_data['model_score'] >= 0.5).astype(int)

false_positives = test_data[(test_data['predicted'] == 1) & (test_data['actual'] == 0)]
false_negatives = test_data[(test_data['predicted'] == 0) & (test_data['actual'] == 1)]

print('false positives:', len(false_positives))
print('false negatives:', len(false_negatives))

false_positives[features + ['model_score', 'actual']].head()

false positives: 1616
false negatives: 4052


,avg_impressions_first_half,avg_position_first_half,avg_ctr_first_half,word_count,content_age_days,model_score,actual
5,14.133333,7.190639,0.006667,2730,39,0.570,0
15,2.666667,12.959184,0.000000,2997,40,0.775,0
24,1.200000,5.574074,0.000000,4017,94,0.675,0
46,4.333333,42.259762,0.000000,2768,62,0.550,0
48,1.800000,7.948718,0.000000,3019,40,0.680,0


In [28]:
# Error analysis
# False positives: 1616, false negatives: 4052
# The model misses more real declines than it wrongly flags, so it leans
# cautious rather than trigger happy

# Looking at the false positive rows, many are quite new pages, around
# 40 to 90 days old, with low impressions and low CTR
# This suggests the model sometimes confuses a new page that naturally
# has low traffic yet with a page that is truly declining
# A future improvement could add a minimum content age before calling
# something a decline, or treat new pages as a separate case entirely

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.